In [ ]:
# old code:

# ENP genesis bounds (0–360° lon)
ENP_BOX = dict(lat_min=0, lat_max=40, lon_min=220, lon_max=280)   # ENP genesis bounds (0–360° lon); 140°W–American coast
# 200–260°E = 160–100°W → includes 100–75°W (Central America coastline)

# CNP genesis bounds (0–360° lon)
CNP_BOX = dict(lat_min=0, lat_max=40, lon_min=180, lon_max=220)

# create column for v^2 needed for ACE calculation
df_tc['v2'] = df_tc['wind']**2

# 6 h step length (days) – not used in ACE sum
DT_DAYS = 6/24.0

# aggregate to storm × year × basin
# ACE = Σ v^2
# MaxWind = max wind for the storm that season

df_agg = (df_tc.groupby(['sid','year','basin'])
              .agg({'v2':'sum','wind':'max'})
              .reset_index())

# rename columns
df_agg = df_agg.rename(columns={'v2':'ACE','wind':'MaxWind'})

# what am I looking at? XXX
print(df_agg)

# calculate annual basin totals/means
annual = (df_agg.groupby(['year','basin'])
                 .agg({'ACE':'sum','MaxWind':'mean'})
                 .reset_index())

# what am I looking at? XXX
print(annual)

# merge ENSO categories
df_merged = pd.merge(annual, enso_jjaso, on='year', how='left')

# add EDA to merged dataframe - XXX - what am I looking at?
print(df_merged)

#############
# read in global best‑track archive (TC data)
ib = xr.open_dataset(ib_path)

# normalize longitudes to 0–360°
ib['lon'] = xr.where(ib['lon']<0, ib['lon']%360, ib['lon'])

# print TC data
print(ib)

# select winds as variable of interest
wind_var = 'usa_wind'

# convert to a tidy table for dataframe operability
df_tc = ib[['sid','time','lat','lon',wind_var]].to_dataframe().reset_index(drop=False)

# rename wind column
df_tc = df_tc.rename(columns={wind_var:'wind'})

# print TC dataframe
print(df_tc)

# check for nans in key columns 
#  extremely high numbers of nans is expected since IBTrACS contains every global basin and many historical records & older records (especially pre‐satellite era) have incomplete lat/lon/wind data
print(df_tc.isnull().sum())

# dataframe cleaning/organizing

# convert to datetime
df_tc['time'] = pd.to_datetime(df_tc['time'], errors='coerce') 

# drop incomplete records
df_tc = df_tc.dropna(subset=['time','lat','lon','wind'])

# extract year and month columns
df_tc['year']  = df_tc['time'].dt.year
df_tc['month'] = df_tc['time'].dt.month

# subset to temporal study window
df_tc = df_tc[(df_tc['year'].between(START_YEAR,END_YEAR)) &
              (df_tc['month'].isin(SEASON_MONTHS))].copy()

# add df_tc EDA
print(df_tc)

# check all nans have been removed
print(df_tc.isnull().sum())

# assign each storm to a single basin by genesis location (first observation per SID)
# first record used as genesis proxy (storms can travel into different basins but here they associate with the basin the originate in)

# groupby storm id and retain the first record per storm as 'origin' to determine  genesis location
origin = df_tc.groupby('sid').first().reset_index()

# initialize classification
origin['basin'] = 'OTHER'

# assign basin by genesis location into Eastern adn Central North Pacific
# ENP genesis
origin.loc[(origin['lat'].between(ENP_BOX['lat_min'],ENP_BOX['lat_max'])) &
           (origin['lon'].between(ENP_BOX['lon_min'],ENP_BOX['lon_max'])),'basin'] = 'ENP'

# CNP genesis
origin.loc[(origin['lat'].between(CNP_BOX['lat_min'],CNP_BOX['lat_max'])) &
           (origin['lon'].between(CNP_BOX['lon_min'],CNP_BOX['lon_max'])),'basin'] = 'CNP'

# broadcast basin label to every 6‑hourly record under the same SID and retain ENP/CNP only
df_tc = df_tc.merge(origin[['sid','basin']], on='sid', how='left')
df_tc = df_tc[df_tc['basin'].isin(['ENP','CNP'])]

# EDA: plot all JJASO fixes to visually QC the spatial domain coverage
#    - color by basin?
#    - plot basin boxes?
#    - clean up figure

# color palette for basins
basin_colors = {'ENP':'forestgreen', 'CNP':'darkorchid'}

fig = plt.figure(figsize=(10,6))
ax  = plt.axes(projection=ccrs.PlateCarree(central_longitude=210))
ax.set_extent([120,300,-20,50], crs=ccrs.PlateCarree())

# add geographic layers
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, zorder=1)
ax.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False)

# plot TC fixes (lon/lat points)
# Plot basin-coded TC fixes
for basin, color in basin_colors.items():
    subset = df_tc[df_tc['basin'] == basin]
    ax.scatter(subset['lon'], subset['lat'], s=4, alpha=0.5, color=color,
               label=f"{basin} fixes", transform=ccrs.PlateCarree())
    
plt.title('All TC Fixes (JJASO 1979–2023, ENP + CNP)', fontsize=12)
plt.legend(loc='upper right')
plt.show()

# INTERPRETATION: 
#
# - Eastern Pacific tracks (green) dominate
# - Central Pacific tracks (purple) are fewer but distinct
# - Storm motion patterns look climatologically reasonable
# - Confirms the correctness of the genesis-based basin assignment

# print number of points/fixes
print(df_tc['basin'].value_counts())

# INTERPRETATION: 
#
# - represents the total number of 6-hourly tropical cyclone fixes (track points) within each basin for the JJASO season from 1979–2023
# - ENP has ~31,000 fixes vs only ~4,300 in the CNP
# - Approximate ratio: ENP produces ~7× more TC fixes than the CNP
# - Because CNP data volume is so much smaller, statistical power is much lower for CNP

#########
# calculate stepwise distances and speeds using haversine formula

# Earth radius (km)
R = 6371

# radians for haversine
lat1 = np.deg2rad(df_tc['lat'])
lat2 = np.deg2rad(df_tc['lat_next'])

dlat = lat2 - lat1
dlon = np.deg2rad(df_tc['lon_next'] - df_tc['lon'])

# haversine a‑term
a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2 

# central angle
c = 2*np.arctan2(np.sqrt(a), np.sqrt(1-a))

# step distance (km)
df_tc['dist_km'] = R * c

# step Δt (hours)
df_tc['dt_hr']   = (df_tc['time_next'] - df_tc['time']).dt.total_seconds()/3600

# step speed (km/h)
df_tc['speed_kmh'] = df_tc['dist_km'] / df_tc['dt_hr']

#########

# recompute genesis - the first row corresponds to the storm’s earliest fix
genesis = df_tc.groupby('sid').first().reset_index()

# print and interpret genesis dataframe below
genesis

# plot genesis locations by ENSO type

fig = plt.figure(figsize=(14,6))
ax  = plt.axes(projection=ccrs.PlateCarree(central_longitude=210))

# set extent for ENP+CNP analyses
ax.set_extent([160, 280, 0, 30], crs=ccrs.PlateCarree())

# add features
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6, zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, zorder=1)
ax.gridlines(draw_labels=True, dms=False, x_inline=False, y_inline=False)

# plot EP vs CP genesis locations
for t, color, label in [
    ('EP', 'forestgreen', 'Genesis during EP El Niño years'),
    ('CP', 'indigo', 'Genesis during CP El Niño years')
]:
    epcp_years = enso_jjaso.loc[enso_jjaso['enso_type']==t, 'year']
    sub = genesis[genesis['year'].isin(epcp_years)]
    ax.scatter(sub['lon'], sub['lat'],
               s=15, alpha=0.6, color=color,
               label=label, transform=ccrs.PlateCarree())

# plot
plt.title('Tropical Cyclone Genesis During EP vs CP El Niño Years (JJASO 1979–2023)')
plt.legend(loc='upper right')
plt.show()

##########

